# Load product data with Auto Loader

Processes newly arrived CSV files on demand and appends them to `retail_demo.bronze.product`.

In [ ]:
from pyspark.sql import functions as F

source_path = "/Volumes/retail_demo/bronze/raw_data/csv/product/"
schema_path = "/Volumes/retail_demo/bronze/raw_data/_autoloader/product/schema/"
checkpoint_path = "/Volumes/retail_demo/bronze/raw_data/_autoloader/product/checkpoint/"
target_table = "retail_demo.bronze.product"

product_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("header", "true")
    .load(source_path)
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file_path"),
        F.col("_metadata.file_name").alias("_source_file_name"),
        F.col("_metadata.file_size").alias("_source_file_size"),
        F.col("_metadata.file_modification_time").alias("_source_file_modified_at"),
        F.current_timestamp().alias("_ingested_at"),
    )
)

query = (
    product_stream.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()
